# Diagnósticos seleccionados de cada intento

Comparamos un bundle nativo revisado con un archivo acotado de asociación a operaciones.
Los mensajes pertenecen al consumidor controlado de Lab; la biblioteca dummy permanece
independiente. No son advertencias nativas de SciPy ni una captura completa.
La comprobación científica y la observación de diagnósticos se muestran por separado.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path("experiments").resolve()))
from inspect_diagnostic_association import answers

workspace = Path("diagnostic-example")
print(sys.executable)

In [ ]:
child = subprocess.run(
    [sys.executable, "experiments/run_diagnostic_association.py", str(workspace)],
    capture_output=True,
    text=True,
    check=True,
)
report = json.loads((workspace / "acceptance.json").read_text())
print(child.stdout.strip())

In [ ]:
assert report["science"]["count"] == 4
assert report["science"]["mean"] == 2.5
assert report["science"]["variance"] == 1.25
print("Comprobación científica independiente:", report["science"])

In [ ]:
current = answers(workspace)
assert current["execution"] == "failed"
assert [row["execution"] for row in current["attempts"]] == ["succeeded", "failed", "succeeded"]
print(json.dumps(current["attempts"], indent=2))

In [ ]:
native = json.loads((workspace / "failure-bundle.json").read_text())
selected = json.loads((workspace / "failure-association.json").read_text())
print("Bundle nativo revisado:", json.dumps(native["events"], indent=2))
print("Asociación acotada:", json.dumps(selected["operations"], indent=2))
assert "message" in native["events"][0]
assert "message" not in selected["operations"][0]["entries"][0]

In [ ]:
artifact = workspace / "failure-association.json"
original = artifact.read_bytes()
artifact.unlink()
missing = answers(workspace)
assert missing["execution"] == "failed"
assert missing["attempts"][1]["diagnostics"] == "unavailable"
artifact.write_bytes(original)
assert answers(workspace) == current
print("Archivo ausente: diagnósticos no disponibles; ejecución conservada.")
print("Proceso interrumpido:", report["interrupted"]["status"])